In [1]:
import wrds
import pandas as pd
import numpy as np
import duckdb as dd
import statsmodels.formula.api as smf
db = wrds.Connection()

Loading library list...
Done


Model Setup and Economic Motivation

We study how BDC market returns relate to NAV while explicitly accounting for the fact that NAV is stale and only updated quarterly. Rather than treating NAV as contemporaneous, we use lagged NAV returns, interpreting NAV as the last observable fundamental signal available to investors at time t. This directly addresses the concern that NAV does not reflect real-time information.

Our approach allows the NAV to price relationship to be state-dependent. Specifically, we examine whether macro conditions—captured by changes in the 10-year Treasury rate (discount rate) and credit spreads (risk premia)—affect how much investors rely on NAV when pricing BDCs.

The interaction terms are central. They test whether the explanatory power of NAV declines when valuation is driven more by discounting and risk premia rather than underlying (and stale) fundamentals.


Does NAVs  explanatory power declines when macro conditions shift investors toward discounting and risk premia instead of stale fundamentals.


![image.png](attachment:image.png)



In [2]:
import pandas as pd

df_event_study = pd.read_csv("data/df_event_study.csv", parse_dates=["trade_date", "rate_date"])

print("Date range:")
print(df_event_study["trade_date"].agg(["min", "max"]))

print("\nTickers:")
print(sorted(df_event_study["ticker"].dropna().unique()))

print("\nObservations by ticker:")
print(df_event_study.groupby("ticker")["trade_date"].count())

print("\nMissing values:")
print(df_event_study[["ticker", "trade_date", "rate_date", "dgs10", "dbaa", "spread"]].isna().sum())

print("\nRows with no rate match:")
print(df_event_study["rate_date"].isna().sum())

print("\nSpread consistency check:")
print((df_event_study["spread"] - (df_event_study["dbaa"] - df_event_study["dgs10"])).abs().max())

print("\nDuplicate ticker-trade_date rows:")
print(df_event_study.duplicated(["ticker", "trade_date"]).sum())

df_event_study.head()


Date range:
min   2016-02-23
max   2024-11-12
Name: trade_date, dtype: datetime64[ns]

Tickers:
['ARCC', 'MAIN']

Observations by ticker:
ticker
ARCC    180
MAIN    180
Name: trade_date, dtype: int64

Missing values:
ticker        0
trade_date    0
rate_date     0
dgs10         0
dbaa          0
spread        0
dtype: int64

Rows with no rate match:
0

Spread consistency check:
4.440892098500626e-16

Duplicate ticker-trade_date rows:
0


,trade_date,permno,ticker,price,ret,vol,shrout,rate_date,dgs10,dbaa,spread,announcement_date,nav,nav_ret,event_date,event_day
0,2016-02-23,90401,ARCC,12.92,0.007015,1505792.0,314347.0,2016-02-23,1.74,5.32,3.58,2015-11-04,16.789579,-0.000500,2016-02-24,-1
1,2016-02-24,90401,ARCC,12.92,0.000000,2100705.0,314347.0,2016-02-24,1.75,5.31,3.56,2016-02-24,16.457393,-0.019785,2016-02-24,0
2,2016-02-25,90401,ARCC,13.39,0.036378,2749160.0,314347.0,2016-02-25,1.71,5.27,3.56,2016-02-24,16.457393,-0.019785,2016-02-24,1
3,2016-02-26,90401,ARCC,13.51,0.008962,2112166.0,314347.0,2016-02-26,1.76,5.32,3.56,2016-02-24,16.457393,-0.019785,2016-02-24,2
4,2016-02-29,90401,ARCC,13.66,0.011103,2084226.0,314347.0,2016-02-29,1.74,5.29,3.55,2016-02-24,16.457393,-0.019785,2016-02-24,3


We analysize df_trade_nav

Interpretation:

NAV is stale and it does not explain return which is what we expected.
    We could include only 

In [4]:
# event day study regression (using ret)

import statsmodels.api as sm
import pandas as pd

event_days = df_event_study.groupby(["ticker", "event_date"])["event_day"].agg(list)
complete = event_days[event_days.apply(lambda x: sorted(x) == [-1, 0, 1, 2, 3])].index

df_reg = (
    df_event_study
    .set_index(["ticker", "event_date"])
    .loc[complete]
    .reset_index()
    .copy()
)

df_reg = pd.get_dummies(df_reg, columns=["event_day"], prefix="day")

X = df_reg[["day_0", "day_1", "day_2", "day_3"]].astype(float)
X = sm.add_constant(X)

y = pd.to_numeric(df_reg["ret"], errors="coerce").astype(float)

valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx]

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.029
Date:                Tue, 14 Apr 2026   Prob (F-statistic):              0.392
Time:                        17:39:40   Log-Likelihood:                 978.16
No. Observations:                 360   AIC:                            -1946.
Df Residuals:                     355   BIC:                            -1927.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0032      0.002      1.662      0.0

In [10]:
# event study using (market return)

import statsmodels.api as sm
import pandas as pd

market_df = pd.read_csv("data/crsp_market_return.csv", parse_dates=["trade_date"])

df_reg_market_return = df_reg.merge(
    market_df[["ticker", "trade_date", "excess_ret"]],
    on=["ticker", "trade_date"],
    how="left"
)

X = df_reg_market_return[["day_0", "day_1", "day_2", "day_3"]].astype(float)
X = sm.add_constant(X)

y = pd.to_numeric(df_reg_market_return["excess_ret"], errors="coerce").astype(float)

valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx]

model = sm.OLS(y, X).fit()
print(model.summary())

df_reg.to_csv("data/reg_df.csv", index=False)
df_reg_market_return.to_csv("data/market_return_df.csv", index=False)

# print(df_reg_market_return["excess_ret"].isna().sum())


                            OLS Regression Results                            
Dep. Variable:             excess_ret   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                    0.8170
Date:                Tue, 14 Apr 2026   Prob (F-statistic):              0.515
Time:                        17:49:40   Log-Likelihood:                 1017.5
No. Observations:                 360   AIC:                            -2025.
Df Residuals:                     355   BIC:                            -2006.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0025      0.002      1.480      0.1

interpretation:
Day -1 return :- Average return on the day before announcement is negligible .07% (Not signnificant)
Day 0 return :- Average return on the day of announcement is .22% (Not significant)
Day 1 return :- Average return jumps by .46% or 46 basis points.
Mean revision :- 0.0%

Is Day 1 return .46% a NAV surprise?
Regress market return (t+1) = nav_ret

Is it the nav_change that caused the jump on return by the .46% i.e. 46 basis points.

In [ ]:
# 1.regress return on day 1 nav_ret

df_day1 = df_event_study[df_event_study['event_day'] == 1].copy()

# 2. Define features: The NAV Surprise
X = df_day1['nav_ret'].astype(float)
X = sm.add_constant(X)

# 3. Define target: The Day 1 Return
y = df_day1['ret'].astype(float)

# 4. Run the OLS
model_sensitivity = sm.OLS(y, X, missing='drop').fit()

print(model_sensitivity.summary())

Interpretation:
    
    1. alpha on day 1 is .63% => book value is does not matter.
    2. NAV is -.14%
    
    We conclude that NAV has no bearing on the price.


In [ ]:
# data integrity #

# 1. Check for Look-Ahead Bias (Did we match with a future NAV?)
future_matches = df_reg[df_reg['trade_date'] < df_reg['announcement_date']]

# 2. Check for missing months per ticker since 2016
# Expecting 108 months per ticker (9 years * 12 months)
obs_count = df_reg.groupby('ticker')['trade_date'].count()

# 3. Check for Nulls in key regression columns
nulls = df_reg[['price', 'nav', 'ret', 'nav_ret']].isnull().sum()

print("--- DATA INTEGRITY TEST RESULTS ---")
print(f"Look-ahead bias records: {len(future_matches)}")
print("\nObservations per Ticker (Expected 108):")
print(obs_count)
print("\nMissing Values in Regression Columns:")
print(nulls)

if len(future_matches) == 0 and (obs_count == 108).all():
    print("\n✅ SUCCESS: No records dropped and date logic is sound.")
else:
    print("\n⚠️ WARNING: Check for missing months or join errors.")



In this sectionm we add 
1. federal reserve 10 year tbill rate
2. credit spread 


In [ ]:
# trade + rate (daily)

query = """
SELECT 
    t.trade_date,
    t.ret,
    t.announcement_date,
    d.dgs10,
    d.dbaa,
    CAST(d.dbaa AS DECIMAL(10,3)) - CAST(d.dgs10 AS DECIMAL(10,3)) AS spread
FROM df_trade_nav_daily t
JOIN df_rate_daily d ON t.trade_date = d.rate_date
"""

df_combined = dd.query(query).to_df()
df_combined